# Lab 4 - Deploy and try the models

You deploy the **instructor's models** - the same pipeline you just ran, on 20x more data - next to
the base model on your `ml.g7.2xlarge` endpoint. Each LoRA adapter is its own **inference
component** sharing the base model's GPU (up to 4 adapters at once). Here you:
1. **Deploy** the instructor's SFT and SFT -> RLVR adapters (shipped in the repo, `adapters/`),
   and your own adapters if your jobs have finished.
2. **Check live outputs** with the deterministic rules (budget, avoid list, item count).
3. **Try them in the Streamlit demo.**

| Model | Trained on | Where it comes from |
|---|---|---|
| Instructor SFT (20K) | 20,000 raw requests | `adapters/v2-sft` |
| Instructor SFT -> RLVR (20K) | 20,000 raw requests | `adapters/v2-sft-rlvr` |
| Your SFT (1K) / Your SFT -> RLVR (1K) | your 1,000 raw requests | your Lab 3 jobs, when they finish |

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath("../src"))
%load_ext autoreload
%autoreload 2
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from octank import config, state
S = config.get_settings("attendee")
ST = state.load(S)
print(f"region={S.region}  bucket={S.bucket}  prefix={S.prefix}  run={S.run_name}")
print(f"role={S.role_arn}")

In [ ]:
from octank import arms, checks, serving
arms.check_handoff(S)               # tells you if SFT -> RLVR needs resuming: arms.resume_rlvr(S)
arms.status(S)                      # your own jobs keep running in the background

## 1. Deploy the instructor's adapters (and yours, if finished)

In [ ]:
ST = state.load(S)
ENDPOINT, BASE_IC = ST["endpoint"], ST["base_ic"]
if ST.get("endpoint_status") != "InService":
    print("waiting for endpoint ...")
    serving.create_endpoint(ENDPOINT, S.role_arn, S.region)
    serving.add_model(ENDPOINT, S.role_arn, S.region)

def deploy_ready():
    ics = dict(state.load(S).get("ics", {}))
    # 1. the instructor's adapters: copy from the repo to your bucket, attach as inference components
    for key, folder in serving.INSTRUCTOR_ADAPTERS.items():
        if key not in ics:
            uri = serving.upload_adapter(folder, S.s3("adapters", folder), S.region)
            ic = f"{ENDPOINT}-{key.replace('_', '-')}"
            serving.add_adapter(ENDPOINT, BASE_IC, ic, uri, S.region)
            ics[key] = ic
    # 2. your own adapters, once their training job has finished
    for arm, a in state.load(S).get("arms", {}).items():
        if a.get("model_package") and arm not in ics:
            ic = f"{ENDPOINT}-{arm.replace('_', '-')}"
            serving.add_adapter(ENDPOINT, BASE_IC, ic, serving.adapter_uri_from_job(a["job"], S.region), S.region)
            ics[arm] = ic
    state.update(S, ics=ics)
    return ics

ics = deploy_ready()        # re-run later to add your own models when they finish training
ics

## 2. Deterministic checks on live outputs

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from octank.prompts import build_prompt
cases = [json.loads(l) for l in open(config.DATA_DIR / "benchmark_cases.jsonl")][:12]
targets = state.load(S).get("ics", {})          # deployed adapters (the base model rarely finishes)

def score(args):
    name, ic, c = args
    out = serving.chat(ENDPOINT, ic, build_prompt(c["request"], c["candidates"]), S.region)
    r = checks.check_text(out["text"], c["request"], c["candidates"])
    return {"model": name, **{k: r[k] for k in ["json_valid", "hard_pass", "within_budget",
            "no_avoid", "item_count_ok"]}, "tokens": (out.get("usage") or {}).get("completion_tokens"),
            "latency_s": out["latency_s"]}

work = [(n, ic, c) for n, ic in targets.items() for c in cases]
with ThreadPoolExecutor(2) as ex:                   # the endpoint streams one request at a time
    res = pd.DataFrame(list(ex.map(score, work)))
res.groupby("model").mean(numeric_only=True).round(3)

## 3. Streamlit demo

Run the cell, then open **`<your Studio URL>/jupyterlab/default/proxy/8501/`** in a new tab. Pick a
random easy / medium / hard / extreme scenario, compare the models side by side, and open the
Catalog page to see the traps each request hides. Your own models show up in **Compare** once you
re-run the deploy cell after they finish; until then the sidebar shows their training status.

In [ ]:
!cd .. && OCTANK_APP_RUN=attendee nohup streamlit run app/app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > work/streamlit.log 2>&1 &
print("open: /jupyterlab/default/proxy/8501/")

## 4. Clean up (stops endpoint billing)

In [ ]:
# serving.delete_all(ST["endpoint"], S.region, confirm_release_endpoint=True)